In [7]:
!pip install -q torch transformers datasets tokenizers \
    sentence-transformers rank-bm25 pytrec-eval-terrier tqdm scipy faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os, sys, logging, json
import numpy as np
import torch

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
sys.path.insert(0, "/content/drive/MyDrive/colbert_ru/code/")
sys.path.insert(0, "/content/drive/MyDrive/colbert_ru")

DRIVE = "/content/drive/MyDrive/colbert_ru"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [4]:
with open(f"{DRIVE}/datasets/corpus.jsonl") as f:
    corpus = [json.loads(line) for line in f]
corpus_texts = [d["text"] for d in corpus]
corpus_ids = [d["id"] for d in corpus]

with open(f"{DRIVE}/datasets/queries.jsonl") as f:
    queries_data = [json.loads(line) for line in f]
queries_dict = {q["id"]: q["text"] for q in queries_data}

qrels = {}
with open(f"{DRIVE}/datasets/qrels.tsv") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) >= 4:
            qid, _, did, rel = parts[:4]
            qrels.setdefault(qid, {})[did] = int(rel)

print(f"Corpus: {len(corpus_texts)} docs, Queries: {len(queries_dict)}, Qrels: {len(qrels)} queries")

Corpus: 35008 docs, Queries: 1252, Qrels: 1252 queries


In [11]:
from evaluate import evaluate_robustness, IRMetrics
from config import EvalConfig

eval_cfg = EvalConfig(
    robustness_typo_rate=0.2,
    robustness_num_augments=4,
    k_values=[1, 5, 10, 20, 100],
)

In [8]:
from benchmark_dense_baselines import build_e5_baseline

e5 = build_e5_baseline(batch_size=64)
e5_build_time = e5.build_index(
    corpus_texts, corpus_ids,
    index_dir=f"{DRIVE}/dense_index_e5_base",
)
print(f"E5-base index built in {e5_build_time:.1f}s, size={e5.index_size_mb():.1f} MB")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/547 [00:00<?, ?it/s]

E5-base index built in 540.0s, size=102.6 MB


In [12]:
TOP_K = 100

rob_e5 = evaluate_robustness(
    retrieve_fn=e5.retrieve_fn(top_k=TOP_K),
    queries=queries_dict,
    qrels=qrels,
    eval_cfg=eval_cfg,
)

print("\n--- E5-base: Robustness degradation (%) ---")
for metric, pct in rob_e5["degradation_pct"].items():
    print(f"  {metric}: -{pct:.1f}%")


--- E5-base: Robustness degradation (%) ---
  map_cut_100: -26.1%
  ndcg_cut_100: -23.5%
  recall_100: -16.0%
  ndcg_cut_5: -26.6%
  recall_5: -25.7%
  ndcg_cut_20: -25.0%
  ndcg_cut_10: -25.8%
  recall_20: -21.6%
  recip_rank: -26.1%
  map_cut_5: -27.0%
  map_cut_1: -27.9%
  ndcg_cut_1: -27.9%
  map_cut_20: -26.3%
  map_cut_10: -26.6%
  recall_10: -23.8%
  recall_1: -27.9%
